# Drive and utilities


In [1]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.4/119.4 kB 2.7 MB/s eta 0:00:00


# Data

In [3]:
#@title Load Data
!pip install numpy pandas scipy scikit-learn sklearn tqdm cudf cupy

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from scipy import stats
import cudf

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Load citations as cudf DataFrame from parquet.
citations_pd = pd.read_pickle("/content/drive/MyDrive/PhD Data/08 Citations/03 Patent citations - raw, filing.pickle")

## Load treated and control data as Pandas DataFrames so embeddings are preserved.
treated = pd.read_pickle("/content/drive/MyDrive/PhD Data/10 Sample - pre final/acquired_patents.pkl")
control = pd.read_pickle("/content/drive/MyDrive/PhD Data/10 Sample - pre final/potential_controls.pkl")

## Remove GAFAM patents
clean_potential_ids = pd.read_csv("/content/drive/MyDrive/PhD Data/10 Sample - pre final/clean_potential_control_ids.csv")
clean_set = set(clean_potential_ids['patent_id'].astype(str))

# Ensure the relevant columns are strings.
treated['patent_id'] = treated['patent_id'].astype(str)
treated['cpc_subclass'] = treated['cpc_subclass'].astype(str)
control['patent_id'] = control['patent_id'].astype(str)
control['cpc_subclass'] = control['cpc_subclass'].astype(str)

## Do not drop the embedding columns now, so we can use them for hybrid matching.

## Remove non-clean controls
control = control[control['patent_id'].isin(clean_set)]
assert len(control) == len(clean_potential_ids)

#--------------------------------------
# Citations
#--------------------------------------
## Rename
citations = cudf.DataFrame.from_pandas(citations_pd)
citations['patent_id'] = citations['patent_id'].astype(str)
citations = citations.rename(columns={'patent_id': 'citedby_patent_id', 'citation_patent_id':'patent_id', 'filing_date':'citation_date'}, inplace = True)

## Remove non-clean controls
control = control[control['patent_id'].isin(clean_set)]
assert len(control) == len(clean_potential_ids)

# TRIM CITATIONS DATA
# Only keep citations for patents that are either treated or control.
treated_ids = set(treated['patent_id'].unique())
control_ids = set(control['patent_id'].unique())
valid_ids = treated_ids.union(control_ids)
citations = citations[citations['patent_id'].isin(valid_ids)]
print("Trimmed citations shape:", citations.shape)


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
Using device: cuda
Trimmed citations shape: (44962852, 4)


In [4]:
#@title Bring Acquisition Type from the DTA
import pandas as pd

# Load and merge
dta = pd.read_stata("/content/drive/MyDrive/PhD Data/09 Acquired patents/04 All patents.dta")
treated = treated.merge(dta[['patent_id', 'acq_type', 'deal_id']], on='patent_id', how='left')

In [5]:
#@title Precompute Treated Patent Quarterly Counts
import cudf
import pandas as pd
import numpy as np
import cudf
import pandas as pd
import numpy as np

# ================================================
# Step 1: Precompute Quarterly Citation Counts for All Patents (GPU version)
# ================================================
# Assume 'citations' is a cuDF DataFrame with columns:
#   - 'patent_id': the cited patent's ID
#   - 'citation_date': a datetime column

# Manually compute a quarter string: "YYYYQX"
citations['citation_date'] = cudf.to_datetime(citations['citation_date'])
citations['year'] = citations['citation_date'].dt.year
citations['month'] = citations['citation_date'].dt.month
# Compute quarter as integer: ((month-1)//3 + 1)
citations['qtr'] = ((citations['month'] - 1) // 3 + 1).astype(str)
citations['citation_quarter'] = citations['year'].astype(str) + "Q" + citations['qtr']

# Group by patent_id and citation_quarter to count citations per quarter.
# cuDF's groupby().size() returns a Series; unstack is not supported, so convert to pandas.
grouped = citations.groupby(['patent_id', 'citation_quarter']).size()
quarterly_counts_pd = grouped.to_pandas().unstack(fill_value=0)

# Build a dictionary mapping patent_id -> {quarter: count}
citation_counts_dict = {}
for pid, row in quarterly_counts_pd.iterrows():
    citation_counts_dict[pid] = row.to_dict()


# ================================================
# Step 2: Precompute Treated Patent Quarterly Vectors
# (Treated data is now a Pandas DataFrame with an 'embedding' column)
# ================================================

# Manually compute the acquisition quarter using pandas datetime methods.
treated['year'] = treated['acq_date'].dt.year
treated['month'] = treated['acq_date'].dt.month
treated['qtr'] = ((treated['month'] - 1) // 3 + 1).astype(str)
treated['acq_quarter'] = treated['year'].astype(str) + "Q" + treated['qtr']

# Build a dictionary mapping each treated patent_id to its pre-treatment quarterly vector.
# For each treated patent, define the four quarters immediately preceding its acquisition quarter.
treated_counts_dict = {}
for i, row in treated.iterrows():
    treated_id = row['patent_id']
    acq_period = pd.Period(row['acq_date'], freq='Q')
    # Pre-treatment quarters: four quarters immediately before acquisition.
    pre_quarters = [str(acq_period - j) for j in range(4, 0, -1)]
    # For each quarter, look up the citation count from citation_counts_dict (defaulting to 0 if missing).
    vec = [citation_counts_dict.get(treated_id, {}).get(q, 0) for q in pre_quarters]
    treated_counts_dict[treated_id] = {'pre_quarters': pre_quarters, 'vector': np.array(vec, dtype=float)}

# For example, print a few treated counts:
print("Example treated counts:")
for pid, info in list(treated_counts_dict.items())[:5]:
    print(pid, info)



Example treated counts:
6829612 {'pre_quarters': ['2007Q1', '2007Q2', '2007Q3', '2007Q4'], 'vector': array([0., 0., 0., 0.])}
5790423 {'pre_quarters': ['2007Q1', '2007Q2', '2007Q3', '2007Q4'], 'vector': array([0., 1., 2., 0.])}
5872712 {'pre_quarters': ['2007Q1', '2007Q2', '2007Q3', '2007Q4'], 'vector': array([0., 0., 2., 0.])}
5926624 {'pre_quarters': ['2007Q1', '2007Q2', '2007Q3', '2007Q4'], 'vector': array([11.,  9., 17., 17.])}
6158005 {'pre_quarters': ['2007Q1', '2007Q2', '2007Q3', '2007Q4'], 'vector': array([1., 0., 2., 0.])}


# Matching for M&A patents

## Matching on both pre-treatment outcomes and BERT embeddings

In [6]:
treated.head()

,patent_id,cpc_subclass,application_year,grant_year,grant_date,acq_date,acq_year,embedding,acq_type,deal_id,year,month,qtr,acq_quarter
0,6829612,G06Q,2001.0,2004,2004-12-07,2008-01-18,2008,"[-0.3726, -0.9355, -0.8286, 0.0538, -0.8154, 0...",M&A,7.0,2008,1,1,2008Q1
1,5790423,H04H,1995.0,1998,1998-08-04,2008-03-16,2008,"[-0.1107, -1.244, -0.2847, 1.446, -0.3142, -0....",M&A,8.0,2008,3,1,2008Q1
2,5872712,G11B,1997.0,1999,1999-02-16,2008-03-16,2008,"[-0.51, -0.565, -0.5425, 1.17, 0.6323, 0.4539,...",M&A,8.0,2008,3,1,2008Q1
3,5926624,G11B,1996.0,1999,1999-07-20,2008-03-16,2008,"[-0.5435, -0.5566, -0.36, 1.595, 0.00869, 1.00...",M&A,8.0,2008,3,1,2008Q1
4,6158005,H04N,1998.0,2000,2000-12-05,2008-03-16,2008,"[-0.4885, 0.2964, -1.458, 0.68, 0.418, 1.155, ...",M&A,8.0,2008,3,1,2008Q1


In [7]:
#@title Get Top Tech M&A patents

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Get M&A patents only
treated_filtered = treated[treated['acq_type'] == 'M&A']

# Get top tech
treated_filtered['pre_treatment_total'] = treated_filtered['patent_id'].apply(
    lambda pid: treated_counts_dict[pid]['vector'].sum() if pid in treated_counts_dict else 0
)
threshold = np.percentile(treated_filtered['pre_treatment_total'], 90)
treated_filtered = treated_filtered[treated_filtered['pre_treatment_total'] >= threshold]



<ipython-input-7-6cb56e327549>:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  treated_filtered['pre_treatment_total'] = treated_filtered['patent_id'].apply(


In [8]:
#@title  Precompute Cosine Distances for Each Treated Patent
# for groups defined by (grant_year, cpc_subclass)
# -------------------------------------------

import torch
import torch.nn.functional as F

# Create a dictionary to store the cosine distance vectors.
# Key: treated_id; Value: cosine distance vector (numpy array) from the treated embedding
# to each candidate control in its group.
cosine_distance_by_treated = {}

# Group treated patents by (grant_year, cpc_subclass)
group_cols_simple = ['grant_year', 'cpc_subclass']
treated_groups_simple = treated_filtered.groupby(group_cols_simple)

for group_key, group in tqdm(treated_groups_simple, total=len(treated_groups_simple), desc="Precompute Cosine Distances"):
    grant_year_val, cpc_subclass_val = group_key
    # For candidate controls, select those that match on grant_year and cpc_subclass.
    candidates = control[(control['grant_year'] == grant_year_val) &
                         (control['cpc_subclass'] == cpc_subclass_val)]
    if candidates.empty:
        continue
    # Convert candidate controls to Pandas.
    candidates_pd = candidates.copy()
    # Precompute candidate embedding tensor on GPU.
    candidate_embeddings = np.stack(candidates_pd['embedding'].values)  # shape: (n_candidates, dim)
    cand_emb = torch.tensor(candidate_embeddings, dtype=torch.float16, device='cuda')

    # For each treated patent in this group, compute and store its cosine distance vector.
    for idx, treated_row in group.iterrows():
        tid = treated_row['patent_id']
        # Get treated embedding as tensor.
        t_embed_np = treated_row['embedding']  # assumed NumPy array
        t_embed = torch.tensor(t_embed_np, dtype=torch.float16, device='cuda').view(1, -1)
        # Compute cosine similarity with all candidate embeddings.
        cos_sim = F.cosine_similarity(t_embed, cand_emb, dim=1)  # shape: (n_candidates,)
        # Cosine distance:
        d_e = 1 - cos_sim.cpu().numpy()
        # Store the candidate cosine distances for this treated patent.
        cosine_distance_by_treated[tid] = d_e


Precompute Cosine Distances: 100%|██████████| 428/428 [01:29<00:00,  4.77it/s]


In [9]:
#@title Settings and functions

import numpy as np
import pandas as pd
import cudf
import cupy as cp
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

import time
import numpy as np
import pandas as pd
import cupy as cp
from tqdm import tqdm
def hybrid_matching_for_lambda(lam, treated_df, control_df, treated_counts_dict, citation_counts_dict):
    """
    Perform hybrid matching using hybrid distance:
      d_H(i,j; lam) = lam * d_c(i,j) + (1-lam) * d_e(i,j)
    where:
      d_c is the Mahalanobis distance based on the 4 pre-treatment quarterly citation vector,
      d_e is 1 - cosine similarity between BERT embeddings.

    Exact matching is performed on grant_year and cpc_subclass by grouping.
    Assumes that a global dictionary 'cosine_distance_by_treated' contains precomputed cosine distance
    vectors for each treated patent.

    Returns a DataFrame of matched pairs.
    """
    import numpy as np
    import pandas as pd
    import cupy as cp
    from tqdm import tqdm

    # Pre-group control_df by (grant_year, cpc_subclass) for fast lookup.
    control_group_dict = {
        key: group for key, group in control_df.groupby(['grant_year', 'cpc_subclass'])
    }

    matches = []
    group_cols = ['acq_quarter', 'grant_year', 'cpc_subclass']
    treated_groups = list(treated_df.groupby(group_cols))

    for group_key, group in tqdm(treated_groups, total=len(treated_groups), desc="Hybrid Matching Groups"):
        acq_quarter, grant_year, cpc_subclass = group_key
        acq_period = pd.Period(acq_quarter, freq='Q')
        # Pre-treatment quarters: Q-6 to Q-3.
        pre_quarters = [str(acq_period - i) for i in range(4, 0, -1)]

        # Retrieve candidate controls quickly from the precomputed dictionary.
        candidates = control_group_dict.get((grant_year, cpc_subclass), pd.DataFrame())
        if candidates.empty:
            continue

        # Extract candidate_ids using a fast list conversion.
        candidate_ids = candidates['patent_id'].tolist()
        # Use list comprehension to build candidate_vectors.
        candidate_vectors = [
            [citation_counts_dict.get(cid, {}).get(q, 0) for q in pre_quarters]
            for cid in candidate_ids
        ]

        # Convert candidate vectors to a CuPy array: shape (n_candidates, 4)
        cp.cuda.Stream.null.synchronize()
        candidate_matrix = cp.array(candidate_vectors, dtype=cp.float64)
        if candidate_matrix.shape[0] > 1:
            cov_matrix = cp.cov(candidate_matrix, rowvar=False)
        else:
            cov_matrix = cp.eye(4, dtype=cp.float64)
        inv_cov = cp.linalg.pinv(cov_matrix)

        # Build arrays for all treated patents in the group.
        treated_ids = []
        treated_vectors = []
        cosine_list = []
        for _, row in group.iterrows():
            tid = row['patent_id']
            # Retrieve precomputed cosine distances for this treated patent.
            d_e = cosine_distance_by_treated.get(tid)
            if d_e is None:
                continue
            treated_ids.append(tid)
            treated_info = treated_counts_dict.get(tid, {'pre_quarters': pre_quarters, 'vector': np.zeros(4)})
            treated_vectors.append(treated_info['vector'])
            cosine_list.append(d_e)  # Each d_e is assumed to be a numpy array (length = n_candidates).

        # If no treated patents in this group have precomputed cosine distances, skip.
        if len(treated_vectors) == 0:
            continue

        # Convert treated vectors into a CuPy array T with shape (n_treated, 4)
        T = cp.array(treated_vectors, dtype=cp.float64)
        # Compute differences: diff has shape (n_treated, n_candidates, 4)
        diff = candidate_matrix[None, :, :] - T[:, None, :]
        # Compute squared Mahalanobis distances: d_c^2 = sum((diff @ inv_cov) * diff, axis=2)
        d_c_sq = cp.sum((diff @ inv_cov) * diff, axis=2)
        d_c = cp.sqrt(d_c_sq)  # shape: (n_treated, n_candidates)
        cp.cuda.Stream.null.synchronize()
        d_c_np = cp.asnumpy(d_c)

        # Stack cosine distances for all treated patents into a matrix (n_treated, n_candidates)
        d_e_mat = np.stack(cosine_list, axis=0)

        # Compute the hybrid distance matrix.
        d_h = lam * d_c_np + (1 - lam) * d_e_mat

        # For each treated patent, find the candidate with the minimum hybrid distance.
        best_indices = np.argmin(d_h, axis=1)

        # Build the results for this group.
        for i, tid in enumerate(treated_ids):
            best_idx = best_indices[i]
            matches.append({
                'treated_id': tid,
                'control_id': candidate_ids[best_idx],
                'treated_vector': treated_vectors[i],
                'control_vector': candidate_vectors[best_idx],
                'mahalanobis_distance': float(d_c_np[i, best_idx]),
                'cosine_distance': float(d_e_mat[i, best_idx]),
                'hybrid_distance': float(d_h[i, best_idx]),
                'pre_quarters': pre_quarters
            })

    return pd.DataFrame(matches)


In [10]:
#@title Run matching and placebo tests

# -------------------------------
# Grid Search Over Lambda and Placebo Estimation
# -------------------------------
lambda_values = np.arange(0, 1.05, 0.05)   # 0, 0.2, ..., 1.0

# Create a dictionary to store matched DataFrames for each lambda.
matched_df_dict = {}

# Add acquisition and grant periods as Periods with quarterly frequency.
treated_filtered['acq_period'] = treated_filtered['acq_date'].apply(lambda d: pd.Period(d, freq='Q'))
treated_filtered['grant_period'] = treated_filtered['grant_date'].apply(lambda d: pd.Period(d, freq='Q'))
# Compute number of quarters between grant and acquisition.
treated_filtered['quarters_between'] = treated_filtered.apply(lambda row: (row['acq_period'] - row['grant_period']).n, axis=1)
# Filter: only include treated patents with at least 4 quarters between grant and acq.
filtered_treated = treated_filtered[treated_filtered['quarters_between'] >= 4]


for lam in lambda_values:
    print(f"Running hybrid matching for lambda = {lam:.2f}")
    # Use a random sample (e.g., 500 treated patents) for quick testing.
    matched_df = hybrid_matching_for_lambda(lam, filtered_treated, control, treated_counts_dict, citation_counts_dict)

    # Save this matched_df for later inspection.
    matched_df_dict[lam] = matched_df.copy()


# -------------------------------
# Save the results to Drive
# -------------------------------

import pickle
with open("/content/drive/MyDrive/PhD Data/11 Matches/13 Hybrid matches - Top Tech actual, MA, filing date.pkl", "wb") as f:
    pickle.dump(matched_df_dict, f)


Running hybrid matching for lambda = 0.00


Hybrid Matching Groups: 100%|██████████| 533/533 [00:19<00:00, 26.98it/s]


Running hybrid matching for lambda = 0.05


Hybrid Matching Groups: 100%|██████████| 533/533 [00:19<00:00, 27.70it/s]


Running hybrid matching for lambda = 0.10


...

Hybrid Matching Groups: 100%|██████████| 533/533 [00:18<00:00, 28.17it/s]


Running hybrid matching for lambda = 0.95


Hybrid Matching Groups: 100%|██████████| 533/533 [00:18<00:00, 28.60it/s]


Running hybrid matching for lambda = 1.00


Hybrid Matching Groups: 100%|██████████| 533/533 [00:18<00:00, 28.63it/s]


## Matching on BERT embeddings only

In [11]:
#@title Cosine Similarity matching

# Assume all_patents is your DataFrame and contains a column 'embedding' with NumPy arrays.
# And assume you have columns: 'patent_id', 'cpc_subclass', 'grant_year', 'treated' (1 for acquired, 0 for control).

matched_pairs_sim_gpu = []

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Group by exact match variables: 'cpc_subclass' and 'grant_year'
grouped_treated = list(filtered_treated.groupby(['cpc_subclass', 'grant_year']))
grouped_controls = control.groupby(['cpc_subclass', 'grant_year'])

for (subclass, gyear), t_group in tqdm(grouped_treated, desc="GPU Cosine Matching Groups", total=len(grouped_treated)):
    try:
        c_group = grouped_controls.get_group((subclass, gyear))
    except KeyError:
        # Skip groups with no available controls
        continue

    # Convert control embeddings to a torch tensor on GPU (they are already float16)
    control_embeddings_np = np.stack(c_group['embedding'].values)  # shape: (n_controls, 1024)
    control_embeddings = torch.tensor(control_embeddings_np, dtype=torch.float16, device=device)
    control_ids = c_group['patent_id'].values

    # Loop over treated patents in this group
    for _, t_row in t_group.iterrows():
        treated_id = t_row['patent_id']
        t_embed_np = t_row['embedding']
        t_embed = torch.tensor(t_embed_np, dtype=torch.float16, device=device).view(1, -1)  # shape: (1, 1024)

        # Compute cosine similarity between t_embed and all control embeddings
        sim = F.cosine_similarity(t_embed, control_embeddings, dim=1)  # returns (n_controls,)
        # Convert similarity to distance (lower is better)
        dist = 1 - sim
        best_idx = torch.argmin(dist).item()
        best_control_id = control_ids[best_idx]

        matched_pairs_sim_gpu.append({
            'treated_id': treated_id,
            'matched_id': best_control_id,
            'cosine_similarity': sim[best_idx].item()  # Save the cosine similarity for the pair
        })

df_sim_matches_gpu = pd.DataFrame(matched_pairs_sim_gpu)
print("GPU-based cosine matching complete!")

# ----- Balance Test for Cosine Matching -----
# Here we simply summarize the cosine similarities across matched pairs.
print("Cosine Matching - Summary of cosine similarities:")
print(df_sim_matches_gpu['cosine_similarity'].describe())


Using device: cuda


GPU Cosine Matching Groups: 100%|██████████| 400/400 [00:03<00:00, 101.91it/s]

GPU-based cosine matching complete!
Cosine Matching - Summary of cosine similarities:
count    1112.000000
mean        0.834512
std         0.042752
min         0.647461
25%         0.810913
50%         0.839844
75%         0.864380
max         0.936523
Name: cosine_similarity, dtype: float64


## Compare the matches when lambda = 0

In [12]:
# Get the hybrid matching results when lambda == 0
hybrid_df = matched_df_dict[0.0].copy()
# Rename the column for clarity
hybrid_df = hybrid_df.rename(columns={'control_id': 'matched_id_hybrid'})

# Get the cosine similarity matching results
cosine_df = df_sim_matches_gpu.copy()
cosine_df = cosine_df.rename(columns={'matched_id': 'matched_id_cosine'})

# Merge the two dataframes on treated_id
merged_df = pd.merge(hybrid_df[['treated_id', 'matched_id_hybrid']],
                     cosine_df[['treated_id', 'matched_id_cosine']],
                     on='treated_id', how='inner')

# Check if the matched controls are identical
merged_df['match_equal'] = merged_df['matched_id_hybrid'] == merged_df['matched_id_cosine']

print(merged_df.head())
print("Percentage of matches that are equal:", merged_df['match_equal'].mean())


  treated_id matched_id_hybrid matched_id_cosine  match_equal
0    5163130           5091868           5091868         True
1    5159632           5113444           5113444         True
2    5271061           5204901           5204901         True
3    5481721           5557776           5557776         True
4    5300373           5336274           5336274         True
Percentage of matches that are equal: 1.0


# Matching for Off Deal patents

## Matching on both pre-treatment outcomes and BERT embeddings

In [13]:
#@title Get Top Tech Off deal patents only

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Get M&A patents only
treated_filtered = treated[treated['acq_type'] == 'Off deal']


# Get top tech
treated_filtered['pre_treatment_total'] = treated_filtered['patent_id'].apply(
    lambda pid: treated_counts_dict[pid]['vector'].sum() if pid in treated_counts_dict else 0
)
threshold = np.percentile(treated_filtered['pre_treatment_total'], 90)
treated_filtered = treated_filtered[treated_filtered['pre_treatment_total'] >= threshold]


<ipython-input-13-2504595e99e5>:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  treated_filtered['pre_treatment_total'] = treated_filtered['patent_id'].apply(


In [14]:
#@title  Precompute Cosine Distances for Each Treated Patent
# for groups defined by (grant_year, cpc_subclass)
# -------------------------------------------

import torch
import torch.nn.functional as F

# Create a dictionary to store the cosine distance vectors.
# Key: treated_id; Value: cosine distance vector (numpy array) from the treated embedding
# to each candidate control in its group.
cosine_distance_by_treated = {}

# Group treated patents by (grant_year, cpc_subclass)
group_cols_simple = ['grant_year', 'cpc_subclass']
treated_groups_simple = treated_filtered.groupby(group_cols_simple)

for group_key, group in tqdm(treated_groups_simple, total=len(treated_groups_simple), desc="Precompute Cosine Distances"):
    grant_year_val, cpc_subclass_val = group_key
    # For candidate controls, select those that match on grant_year and cpc_subclass.
    candidates = control[(control['grant_year'] == grant_year_val) &
                         (control['cpc_subclass'] == cpc_subclass_val)]
    if candidates.empty:
        continue
    # Convert candidate controls to Pandas.
    candidates_pd = candidates.copy()
    # Precompute candidate embedding tensor on GPU.
    candidate_embeddings = np.stack(candidates_pd['embedding'].values)  # shape: (n_candidates, dim)
    cand_emb = torch.tensor(candidate_embeddings, dtype=torch.float16, device='cuda')

    # For each treated patent in this group, compute and store its cosine distance vector.
    for idx, treated_row in group.iterrows():
        tid = treated_row['patent_id']
        # Get treated embedding as tensor.
        t_embed_np = treated_row['embedding']  # assumed NumPy array
        t_embed = torch.tensor(t_embed_np, dtype=torch.float16, device='cuda').view(1, -1)
        # Compute cosine similarity with all candidate embeddings.
        cos_sim = F.cosine_similarity(t_embed, cand_emb, dim=1)  # shape: (n_candidates,)
        # Cosine distance:
        d_e = 1 - cos_sim.cpu().numpy()
        # Store the candidate cosine distances for this treated patent.
        cosine_distance_by_treated[tid] = d_e


Precompute Cosine Distances: 100%|██████████| 247/247 [00:52<00:00,  4.74it/s]


In [15]:
#@title Settings and functions

import numpy as np
import pandas as pd
import cudf
import cupy as cp
import torch
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt

import time
import numpy as np
import pandas as pd
import cupy as cp
from tqdm import tqdm
def hybrid_matching_for_lambda(lam, treated_df, control_df, treated_counts_dict, citation_counts_dict):
    """
    Perform hybrid matching using hybrid distance:
      d_H(i,j; lam) = lam * d_c(i,j) + (1-lam) * d_e(i,j)
    where:
      d_c is the Mahalanobis distance based on the 4 pre-treatment quarterly citation vector,
      d_e is 1 - cosine similarity between BERT embeddings.

    Exact matching is performed on grant_year and cpc_subclass by grouping.
    Assumes that a global dictionary 'cosine_distance_by_treated' contains precomputed cosine distance
    vectors for each treated patent.

    Returns a DataFrame of matched pairs.
    """
    import numpy as np
    import pandas as pd
    import cupy as cp
    from tqdm import tqdm

    # Pre-group control_df by (grant_year, cpc_subclass) for fast lookup.
    control_group_dict = {
        key: group for key, group in control_df.groupby(['grant_year', 'cpc_subclass'])
    }

    matches = []
    group_cols = ['acq_quarter', 'grant_year', 'cpc_subclass']
    treated_groups = list(treated_df.groupby(group_cols))

    for group_key, group in tqdm(treated_groups, total=len(treated_groups), desc="Hybrid Matching Groups"):
        acq_quarter, grant_year, cpc_subclass = group_key
        acq_period = pd.Period(acq_quarter, freq='Q')
        # Pre-treatment quarters: Q-6 to Q-3.
        pre_quarters = [str(acq_period - i) for i in range(4, 0, -1)]

        # Retrieve candidate controls quickly from the precomputed dictionary.
        candidates = control_group_dict.get((grant_year, cpc_subclass), pd.DataFrame())
        if candidates.empty:
            continue

        # Extract candidate_ids using a fast list conversion.
        candidate_ids = candidates['patent_id'].tolist()
        # Use list comprehension to build candidate_vectors.
        candidate_vectors = [
            [citation_counts_dict.get(cid, {}).get(q, 0) for q in pre_quarters]
            for cid in candidate_ids
        ]

        # Convert candidate vectors to a CuPy array: shape (n_candidates, 4)
        cp.cuda.Stream.null.synchronize()
        candidate_matrix = cp.array(candidate_vectors, dtype=cp.float64)
        if candidate_matrix.shape[0] > 1:
            cov_matrix = cp.cov(candidate_matrix, rowvar=False)
        else:
            cov_matrix = cp.eye(4, dtype=cp.float64)
        inv_cov = cp.linalg.pinv(cov_matrix)

        # Build arrays for all treated patents in the group.
        treated_ids = []
        treated_vectors = []
        cosine_list = []
        for _, row in group.iterrows():
            tid = row['patent_id']
            # Retrieve precomputed cosine distances for this treated patent.
            d_e = cosine_distance_by_treated.get(tid)
            if d_e is None:
                continue
            treated_ids.append(tid)
            treated_info = treated_counts_dict.get(tid, {'pre_quarters': pre_quarters, 'vector': np.zeros(4)})
            treated_vectors.append(treated_info['vector'])
            cosine_list.append(d_e)  # Each d_e is assumed to be a numpy array (length = n_candidates).

        # If no treated patents in this group have precomputed cosine distances, skip.
        if len(treated_vectors) == 0:
            continue

        # Convert treated vectors into a CuPy array T with shape (n_treated, 4)
        T = cp.array(treated_vectors, dtype=cp.float64)
        # Compute differences: diff has shape (n_treated, n_candidates, 4)
        diff = candidate_matrix[None, :, :] - T[:, None, :]
        # Compute squared Mahalanobis distances: d_c^2 = sum((diff @ inv_cov) * diff, axis=2)
        d_c_sq = cp.sum((diff @ inv_cov) * diff, axis=2)
        d_c = cp.sqrt(d_c_sq)  # shape: (n_treated, n_candidates)
        cp.cuda.Stream.null.synchronize()
        d_c_np = cp.asnumpy(d_c)

        # Stack cosine distances for all treated patents into a matrix (n_treated, n_candidates)
        d_e_mat = np.stack(cosine_list, axis=0)

        # Compute the hybrid distance matrix.
        d_h = lam * d_c_np + (1 - lam) * d_e_mat

        # For each treated patent, find the candidate with the minimum hybrid distance.
        best_indices = np.argmin(d_h, axis=1)

        # Build the results for this group.
        for i, tid in enumerate(treated_ids):
            best_idx = best_indices[i]
            matches.append({
                'treated_id': tid,
                'control_id': candidate_ids[best_idx],
                'treated_vector': treated_vectors[i],
                'control_vector': candidate_vectors[best_idx],
                'mahalanobis_distance': float(d_c_np[i, best_idx]),
                'cosine_distance': float(d_e_mat[i, best_idx]),
                'hybrid_distance': float(d_h[i, best_idx]),
                'pre_quarters': pre_quarters
            })

    return pd.DataFrame(matches)


In [16]:
#@title Run matching and placebo tests

# -------------------------------
# Grid Search Over Lambda and Placebo Estimation
# -------------------------------
lambda_values = np.arange(0, 1.05, 0.05)   # 0, 0.2, ..., 1.0

# Create a dictionary to store matched DataFrames for each lambda.
matched_df_dict = {}

# Add acquisition and grant periods as Periods with quarterly frequency.
treated_filtered['acq_period'] = treated_filtered['acq_date'].apply(lambda d: pd.Period(d, freq='Q'))
treated_filtered['grant_period'] = treated_filtered['grant_date'].apply(lambda d: pd.Period(d, freq='Q'))
# Compute number of quarters between grant and acquisition.
treated_filtered['quarters_between'] = treated_filtered.apply(lambda row: (row['acq_period'] - row['grant_period']).n, axis=1)
# Filter: only include treated patents with at least 4 quarters between grant and acq.
filtered_treated = treated_filtered[treated_filtered['quarters_between'] >= 4]


for lam in lambda_values:
    print(f"Running hybrid matching for lambda = {lam:.2f}")
    # Use a random sample (e.g., 500 treated patents) for quick testing.
    matched_df = hybrid_matching_for_lambda(lam, filtered_treated, control, treated_counts_dict, citation_counts_dict)

    # Save this matched_df for later inspection.
    matched_df_dict[lam] = matched_df.copy()


# -------------------------------
# Save the results to Drive
# -------------------------------

import pickle
with open("/content/drive/MyDrive/PhD Data/11 Matches/13 Hybrid matches - Top Tech actual, OD, filing date.pkl", "wb") as f:
    pickle.dump(matched_df_dict, f)


Running hybrid matching for lambda = 0.00


Hybrid Matching Groups: 100%|██████████| 340/340 [00:09<00:00, 36.30it/s]


Running hybrid matching for lambda = 0.05


Hybrid Matching Groups: 100%|██████████| 340/340 [00:09<00:00, 36.33it/s]


Running hybrid matching for lambda = 0.10


...

Hybrid Matching Groups: 100%|██████████| 340/340 [00:09<00:00, 36.29it/s]


Running hybrid matching for lambda = 0.95


Hybrid Matching Groups: 100%|██████████| 340/340 [00:08<00:00, 38.27it/s]


Running hybrid matching for lambda = 1.00


Hybrid Matching Groups: 100%|██████████| 340/340 [00:08<00:00, 38.32it/s]


## Matching on BERT embeddings only

In [17]:
#@title Cosine Similarity matching

# Assume all_patents is your DataFrame and contains a column 'embedding' with NumPy arrays.
# And assume you have columns: 'patent_id', 'cpc_subclass', 'grant_year', 'treated' (1 for acquired, 0 for control).

matched_pairs_sim_gpu = []

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Group by exact match variables: 'cpc_subclass' and 'grant_year'
grouped_treated = list(filtered_treated.groupby(['cpc_subclass', 'grant_year']))
grouped_controls = control.groupby(['cpc_subclass', 'grant_year'])

for (subclass, gyear), t_group in tqdm(grouped_treated, desc="GPU Cosine Matching Groups", total=len(grouped_treated)):
    try:
        c_group = grouped_controls.get_group((subclass, gyear))
    except KeyError:
        # Skip groups with no available controls
        continue

    # Convert control embeddings to a torch tensor on GPU (they are already float16)
    control_embeddings_np = np.stack(c_group['embedding'].values)  # shape: (n_controls, 1024)
    control_embeddings = torch.tensor(control_embeddings_np, dtype=torch.float16, device=device)
    control_ids = c_group['patent_id'].values

    # Loop over treated patents in this group
    for _, t_row in t_group.iterrows():
        treated_id = t_row['patent_id']
        t_embed_np = t_row['embedding']
        t_embed = torch.tensor(t_embed_np, dtype=torch.float16, device=device).view(1, -1)  # shape: (1, 1024)

        # Compute cosine similarity between t_embed and all control embeddings
        sim = F.cosine_similarity(t_embed, control_embeddings, dim=1)  # returns (n_controls,)
        # Convert similarity to distance (lower is better)
        dist = 1 - sim
        best_idx = torch.argmin(dist).item()
        best_control_id = control_ids[best_idx]

        matched_pairs_sim_gpu.append({
            'treated_id': treated_id,
            'matched_id': best_control_id,
            'cosine_similarity': sim[best_idx].item()  # Save the cosine similarity for the pair
        })

df_sim_matches_gpu = pd.DataFrame(matched_pairs_sim_gpu)
print("GPU-based cosine matching complete!")

# ----- Balance Test for Cosine Matching -----
# Here we simply summarize the cosine similarities across matched pairs.
print("Cosine Matching - Summary of cosine similarities:")
print(df_sim_matches_gpu['cosine_similarity'].describe())


Using device: cuda


GPU Cosine Matching Groups: 100%|██████████| 229/229 [00:02<00:00, 92.43it/s]

GPU-based cosine matching complete!
Cosine Matching - Summary of cosine similarities:
count    867.000000
mean       0.857282
std        0.042372
min        0.622559
25%        0.834473
50%        0.860840
75%        0.882324
max        1.000000
Name: cosine_similarity, dtype: float64


## Compare the matches when lambda = 0

In [18]:
# Get the hybrid matching results when lambda == 0
hybrid_df = matched_df_dict[0.0].copy()
# Rename the column for clarity
hybrid_df = hybrid_df.rename(columns={'control_id': 'matched_id_hybrid'})

# Get the cosine similarity matching results
cosine_df = df_sim_matches_gpu.copy()
cosine_df = cosine_df.rename(columns={'matched_id': 'matched_id_cosine'})

# Merge the two dataframes on treated_id
merged_df = pd.merge(hybrid_df[['treated_id', 'matched_id_hybrid']],
                     cosine_df[['treated_id', 'matched_id_cosine']],
                     on='treated_id', how='inner')

# Check if the matched controls are identical
merged_df['match_equal'] = merged_df['matched_id_hybrid'] == merged_df['matched_id_cosine']

print(merged_df.head())
print("Percentage of matches that are equal:", merged_df['match_equal'].mean())


  treated_id matched_id_hybrid matched_id_cosine  match_equal
0    5581703           5577103           5577103         True
1    5603058           5664226           5664226         True
2    5668948           5652615           5652615         True
3    5696905           5596705           5596705         True
4    5761417           5721956           5721956         True
Percentage of matches that are equal: 1.0
